# 3D inductive field

`InductiveField3D` solves the time-harmonic magnetoquasistatic field of coils and conductors
in 3D, on the edges of a Cartesian grid (`EdgeGrid`), for geometries without a symmetry,
such as several drivers with their shields. The discretisation is in
[3D field discretisation](/wellbalanced/development/field-3d/). This tutorial checks it on
a long solenoid in a conducting box, whose inductance per length is known in closed form:
$L' = \mu_0 n^2 A_s (1 - A_s / A_\mathrm{box})$.

In [ ]:
import numpy as np
import plasma_plots  # noqa: F401, the .plasma accessor
import xarray as xr
from scipy.constants import mu_0
from simgeometry import Cylinder

from wellbalanced.spatial import Coil3D, EdgeGrid, InductiveField3D

half, radius, turns, depth = 0.1, 0.05, 20.0, 0.02  # m, m, 1/m, m
box_area = (2 * half) ** 2
exact = mu_0 * turns**2 * np.pi * radius**2 * (1 - np.pi * radius**2 / box_area)


def solenoid(cells):
    """A solenoid along z through a box, periodic along z (an infinite solenoid)."""
    grid = EdgeGrid(
        (-half, -half, 0.0),
        (half, half, depth),
        (cells, cells, 2),
        ("dirichlet", "dirichlet", "periodic"),
    )
    coil = Coil3D(Cylinder((0, 0, -1), (0, 0, 1), radius), (0, 0, 1), turns)
    return InductiveField3D(grid, frequency=1e3, coils=[coil])


model = solenoid(20)
solution = model.solve(1.0)  # 1 A
solution.inductance / depth, exact

The coil is a current sheet on the surface of its region, `J = curl T`; on the grid the
circle is staircased, so the error shrinks with the cells:

In [ ]:
cells = [10, 20, 40]
error = xr.DataArray(
    [abs(solenoid(n).solve(1.0).inductance / depth / exact - 1) for n in cells],
    coords={"cells": ("cells", cells, {"long_name": "cells across the box"})},
    name="error",
    attrs={"long_name": "relative error of L'"},
)
error.plasma.plot.convergence()

Conductors of a given conductivity (`Conductor3D`), good conductors as surface impedances
(`ImpedanceConductor3D`) and a background conductivity such as a cold plasma's complex
$\sigma$ (`conductivity=`) add their losses to the solution: `solution.loss`,
`solution.conductor_losses` and `solution.resistance`.

## Coils of real wire

A coil made of copper tube is an `ImpedanceConductor3D` (its inside drops out, Leontovich's
$E_t = Z_s H \times n$ on its surface) driven by a `Feed3D`: the coil current flows along
grid edges across a gap between the coil's terminals, from inside the metal of one
terminal to inside the other, and on the metal's surface everywhere else. The impedance
the terminals see then includes the coil's own Joule loss and its proximity effect.

The check is a coaxial line of copper, inner radius $a$ and outer $b$, shorted at the far
end and fed across a gap at the near one. Per metre,
$Z' = i\omega\frac{\mu_0}{2\pi}\ln\frac{b}{a} + \frac{Z_s}{2\pi}\left(\frac1a + \frac1b\right)$;
the difference of two lengths removes the ends.

In [ ]:
from simgeometry import Cuboid, Tube

from wellbalanced.spatial import Feed3D, ImpedanceConductor3D

copper, frequency, a, b, gap = 5.8e7, 1e6, 0.004, 0.012, 0.002


def coax(length, h):
    """The line from z = 0 (fed) to z = length (shorted by the box's wall)."""
    n = round(0.03 / h)
    grid = EdgeGrid(
        (-0.015, -0.015, 0), (0.015, 0.015, length), (n, n, round(length / h))
    )
    inner = Cylinder((0, 0, gap), (0, 0, length + 1), a)
    outer = Tube((0, 0, -1), (0, 0, length + 1), b, b + 0.002)
    return InductiveField3D(
        grid,
        frequency=frequency,
        feeds=[Feed3D((0, 0, 0), (0, 0, gap + 0.003))],
        impedance_conductors=[
            ImpedanceConductor3D([inner], copper, "inner"),
            ImpedanceConductor3D([outer], copper, "outer"),
        ],
    )


z_s = ImpedanceConductor3D((), copper).surface_impedance(frequency)
exact = 2j * np.pi * frequency * mu_0 / (2 * np.pi) * np.log(b / a) + z_s / (
    2 * np.pi
) * (1 / a + 1 / b)
rows = []
for h in (0.002, 0.001):
    short, long = (
        coax(length, h).solve(1.0, solver="iterative") for length in (0.008, 0.016)
    )
    z = (long.impedance - short.impedance) / 0.008
    rows.append((h, z.real / exact.real - 1, z.imag / exact.imag - 1))
xr.Dataset(
    {
        "resistance_error": ("h", [r[1] for r in rows], {"long_name": "error of R'"}),
        "reactance_error": ("h", [r[2] for r in rows], {"long_name": "error of X'"}),
    },
    coords={"h": ("h", [r[0] for r in rows], {"units": "m"})},
).to_dataframe()

The reactance converges; the resistance is within a few per cent and first order: the
surface loss takes each cut face's average $H$, which here varies as $1/r$ across the cell.

## Slotted cylinder

The test that decided the discretisation: a copper-like tube (6 mm wall) with four axial
4 mm slits in a solenoid, at 500 kHz. In the z-invariant cut the cut cells match NGSolve
(Nédélec elements on meshes fitted to the slits, the same surface impedance) to below
0.1 %, while staircased cells are 20 % off however fine the grid; the finite tube in 3D is
in [3D field discretisation](/wellbalanced/development/field-3d/).

In [ ]:
ngsolve = 0.6682538180678069  # W/m at 1 A, the whole cross-section


def slotted(cells, cut):
    """A quarter of the cross-section, z-periodic, two cells long."""
    h = 0.06 / cells
    grid = EdgeGrid(
        (0, 0, 0),
        (0.06, 0.06, 2 * h),
        (cells, cells, 2),
        ("dirichlet",) * 2 + ("periodic",),
    )
    tube = (
        Tube((0, 0, -1), (0, 0, 1), 0.030, 0.036)
        - Cuboid((0, 0, 0), (1, 0.004, 4))
        - Cuboid((0, 0, 0), (0.004, 1, 4))
    )
    model = InductiveField3D(
        grid,
        frequency=5e5,
        coils=[Coil3D(Cuboid((0, 0, 0), (0.09, 0.09, 4)), (0, 0, 1), 100.0)],
        impedance_conductors=[ImpedanceConductor3D([tube], 1e6)],
        cut_cells=cut,
    )
    return 4 * model.solve(1.0).loss / (2 * h) / ngsolve - 1


cells = [12, 24, 48]
xr.Dataset(
    {
        "cut_cells": ("cells", [slotted(n, True) for n in cells]),
        "staircase": ("cells", [slotted(n, False) for n in cells]),
    },
    coords={"cells": cells},
).to_dataframe()